# 거상(Gersang) 몬스터·드랍아이템 YOLO 학습 — 원클릭 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhleee/gersang-db/blob/main/notebooks/gersang_yolo_training.ipynb)

위에서부터 **▶ 버튼만 차례로 누르면** 학습 → 평가 → 실제 사냥 화면 추론 → ONNX 변환까지 끝납니다.

| 항목 | 내용 |
|---|---|
| 클래스 | 0 imugi(이무기), 1 yeomra(염라대왕), 2 docheol(어린도철), 3 nue(누에), 4 sammun(중독된삼문), 5 item_drop(드랍아이템) |
| 데이터 | 합성 1,000장(640×640, train 800 / val 200) — `tools/generate_yolo_dataset.py` 가 인게임 스프라이트+실제 배경으로 생성 |
| 모델 | YOLOv8n 전이학습 (Colab T4 GPU 기준 50 epoch ≈ 10~15분) |
| 결과물 | `best.pt`, `best.onnx`, 학습 곡선·혼동행렬, 실제 사냥 화면 예측 이미지 |

> 데이터셋을 미리 압축본으로 받아 쓰려면 로컬에서 `tools/generate_yolo_dataset.py` 로 만든 `gersang_yolo_dataset.zip` 을
> 이 노트북 파일창에 업로드한 뒤 셀 3의 `USE_UPLOADED_ZIP = True` 로 바꿔 실행하면 됩니다.


## 1. GPU 확인
학습 전에 어떤 GPU가 붙었는지 확인합니다. (무료 T4면 충분합니다)

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Ultralytics 설치
YOLOv8/YOLO11 학습·추론·변환을 한 번에 제공하는 패키지입니다.

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__)


## 3. 저장소 클론 + 데이터셋 준비

에셋(스프라이트·배경)이 저장소에 들어 있으므로, Colab에서 **데이터셋을 직접 생성**합니다.
(1,000장 생성에 CPU 기준 2~3분, GPU 런타임에서는 더 빠릅니다)

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/jhleee/gersang-db.git"
USE_UPLOADED_ZIP = False   # 압축본을 업로드해 쓰려면 True
DATA_ROOT = "/content/data"

if not pathlib.Path("/content/gersang-db").exists():
    !git clone -q $REPO_URL /content/gersang-db
%cd /content/gersang-db

if USE_UPLOADED_ZIP:
    # 노트북 파일창에 올린 gersang_yolo_dataset.zip 을 사용
    !mkdir -p /content/data && unzip -q -o gersang_yolo_dataset.zip -d /content/data
else:
    # 저장소의 에셋으로 합성 데이터셋 생성 (seed 고정 -> 항상 같은 결과)
    !python tools/generate_yolo_dataset.py --out {DATA_ROOT}/gersang_yolo --images 1000 --size 640 --preview 16

DATA = f"{DATA_ROOT}/gersang_yolo"
# data.yaml 의 path 를 이 런타임 경로로 고정
yaml_path = pathlib.Path(DATA) / "data.yaml"
txt = yaml_path.read_text()
lines = [l for l in txt.splitlines() if not l.startswith("path:")]
yaml_path.write_text(f"path: {DATA}\n" + "\n".join(lines) + "\n")

print(open(yaml_path).read())
print("이미지:", len(os.listdir(f"{DATA}/images/train")), "train /", len(os.listdir(f"{DATA}/images/val")), "val")


## 4. YOLOv8n 전이학습

- `yolov8n.pt` (COCO 사전학습 가중치)에서 시작해 거상 6클래스로 미세조정합니다.
- 무료 T4 기준 `epochs=50, imgsz=640, batch=16` 이면 10~15분입니다.
- 학습이 끝나면 `runs/detect/gersang/weights/best.pt` 가 만들어집니다.

In [ ]:
!yolo detect train \
  data={DATA}/data.yaml \
  model=yolov8n.pt \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  patience=15 \
  project=runs/detect \
  name=gersang \
  exist_ok=True


## 5. 학습 결과 평가 · 시각화

- 검증셋(200장) 지표: precision / recall / mAP50 / mAP50-95
- `results.png`(손실·mAP 곡선), `confusion_matrix.png`, `val_batch*.jpg`(예측 시각화)를 함께 보여줍니다.

In [ ]:
from IPython.display import Image as IPyImage, display
import glob, os

RUN = "runs/detect/gersang"
!yolo detect val model={RUN}/weights/best.pt data={DATA}/data.yaml imgsz=640

for name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    p = os.path.join(RUN, name)
    if os.path.exists(p):
        print("\n###", name)
        display(IPyImage(filename=p, width=880))

for p in sorted(glob.glob(os.path.join(RUN, "val_batch0_pred.jpg")))[:1]:
    print("\n### 검증셋 예측 예시")
    display(IPyImage(filename=p, width=880))


## 6. 실제 거상 사냥 화면 추론 (검수)

`test_images/` 에는 웹에서 수집한 실제 인게임 사냥 화면 13장이 들어 있습니다.
학습에 쓰지 않은 이미지이므로, 합성 데이터로 배운 모델이 실화면에서 얼마나 잡아내는지 바로 확인할 수 있습니다.

In [ ]:
!yolo detect predict \
  model={RUN}/weights/best.pt \
  source=test_images \
  imgsz=640 \
  conf=0.25 \
  save=True \
  save_txt=True \
  project=runs/detect \
  name=real_check

import glob
from IPython.display import Image as IPyImage, display
outs = sorted(glob.glob("runs/detect/real_check/*.jpg"))
print(len(outs), "장 저장됨")
for p in outs[:6]:
    print(p)
    display(IPyImage(filename=p, width=720))


## 7. 가중치 저장 · ONNX 경량화
`best.pt`(재학습용)와 `best.onnx`(실시간 추론·앱 포팅용)를 함께 받습니다.

In [ ]:
!yolo export model={RUN}/weights/best.pt format=onnx imgsz=640 opset=12 simplify=True

import shutil, os
from google.colab import files
shutil.copy(f"{RUN}/weights/best.pt", "/content/gersang_best.pt")
onnx = f"{RUN}/weights/best.onnx"
if os.path.exists(onnx):
    shutil.copy(onnx, "/content/gersang_best.onnx")
    files.download("/content/gersang_best.onnx")
files.download("/content/gersang_best.pt")
print("best.pt / best.onnx 다운로드 시작")


---
### 다음 단계 아이디어
- 실사냥 화면을 100~200장 라벨링해 val에 섞으면 실화면 정확도가 크게 올라갑니다(합성→실사 도메인 갭).
- `test_images/` 처럼 UI가 많은 화면은 화면 영역만 크롭해 추론하면 오탐이 줄어듭니다.
- 프레임 3~5장 다수결(보팅)이나 추적기(BoT-SORT)를 붙이면 순간 오탐이 사라집니다.